# Phase 2: Unsupervised Engineering

**Hybrid Feature Engineering Pipeline — Phase 2**

- **4a. K-Means Clustering:** Fit on train, predict on test → One-Hot Encode cluster IDs
- **4b. PCA on Hitting Stats:** Fit on train, transform test → Principal component scores

Leakage barrier: all unsupervised models fit on train only.

In [3]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

ARTIFACT_DIR = 'artifacts'

# Load Phase 1 outputs
train = pd.read_csv(os.path.join(ARTIFACT_DIR, 'train_phase1.csv'))
test = pd.read_csv(os.path.join(ARTIFACT_DIR, 'test_phase1.csv'))
col_info = joblib.load(os.path.join(ARTIFACT_DIR, 'column_info.pkl'))

feature_cols = col_info['feature_cols']
meta_cols = col_info['meta_cols']

print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')

Train: (1812, 75)
Test:  (453, 69)


## 4a. K-Means Clustering — Team Archetypes

Select a subset of relevant stats (R, RA, HR, ERA, FP) to cluster teams into archetypes.
These are already scaled from Phase 1.

In [4]:
# Features for clustering — core team profile stats (already scaled)
cluster_features = ['R', 'RA', 'HR', 'ERA', 'FP']

print('Clustering features:', cluster_features)
print(f'Train sample stats (scaled):')
train[cluster_features].describe().round(3)

Clustering features: ['R', 'RA', 'HR', 'ERA', 'FP']
Train sample stats (scaled):


,R,RA,HR,ERA,FP
count,1812.000,1812.000,1812.000,1812.000,1812.000
mean,-0.000,-0.000,0.000,-0.000,-0.000
std,1.000,1.000,1.000,1.000,1.000
min,-3.199,-3.114,-2.073,-3.167,-4.822
25%,-0.647,-0.672,-0.672,-0.626,-0.492
50%,-0.003,-0.028,0.083,-0.005,0.251
75%,0.618,0.626,0.712,0.632,0.746
max,3.561,4.874,2.599,4.158,1.859


In [5]:
# Determine optimal K using inertia (elbow method) and silhouette
from sklearn.metrics import silhouette_score

K_range = range(2, 9)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(train[cluster_features])
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(train[cluster_features], labels))
    print(f'K={k}: inertia={km.inertia_:.1f}, silhouette={silhouettes[-1]:.4f}')

print(f'\nBest silhouette: K={list(K_range)[np.argmax(silhouettes)]} '
      f'(score={max(silhouettes):.4f})')

K=2: inertia=5996.5, silhouette=0.2934
K=3: inertia=4662.1, silhouette=0.2719
K=4: inertia=3813.1, silhouette=0.2797
K=5: inertia=3331.6, silhouette=0.2430
K=6: inertia=2998.9, silhouette=0.2407
K=7: inertia=2751.7, silhouette=0.2503
K=8: inertia=2533.4, silhouette=0.2419

Best silhouette: K=2 (score=0.2934)


In [6]:
# Fit K-Means with chosen K
N_CLUSTERS = 4  # adjust based on elbow/silhouette above

kmeans = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42)

# FIT + PREDICT on train
train['cluster_id'] = kmeans.fit_predict(train[cluster_features])

# PREDICT ONLY on test (using train-fitted centroids)
test['cluster_id'] = kmeans.predict(test[cluster_features])

print(f'Train cluster distribution:')
print(train['cluster_id'].value_counts().sort_index())
print(f'\nTest cluster distribution:')
print(test['cluster_id'].value_counts().sort_index())

Train cluster distribution:
cluster_id
0    546
1    197
2    711
3    358
Name: count, dtype: int64

Test cluster distribution:
cluster_id
0    120
1     52
2    186
3     95
Name: count, dtype: int64


In [7]:
# One-Hot Encode cluster IDs
for i in range(N_CLUSTERS):
    col_name = f'is_cluster_{i}'
    train[col_name] = (train['cluster_id'] == i).astype(int)
    test[col_name] = (test['cluster_id'] == i).astype(int)

# Drop the raw cluster_id column
train.drop(columns=['cluster_id'], inplace=True)
test.drop(columns=['cluster_id'], inplace=True)

cluster_ohe_cols = [f'is_cluster_{i}' for i in range(N_CLUSTERS)]
print(f'Added {len(cluster_ohe_cols)} cluster OHE columns: {cluster_ohe_cols}')
print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')

Added 4 cluster OHE columns: ['is_cluster_0', 'is_cluster_1', 'is_cluster_2', 'is_cluster_3']
Train shape: (1812, 79)
Test shape:  (453, 73)


## 4b. PCA on Correlated Hitting Stats

Reduce multicollinearity among correlated offensive stats (H, 2B, 3B, HR, BB) into principal components.

In [8]:
# Correlated hitting features for PCA (already scaled)
pca_features = ['H', '2B', '3B', 'HR', 'BB']

# Check correlations to confirm multicollinearity
print('Correlation matrix (train):')
print(train[pca_features].corr().round(3))

Correlation matrix (train):
        H     2B     3B     HR     BB
H   1.000  0.720  0.083  0.391  0.252
2B  0.720  1.000 -0.221  0.528  0.301
3B  0.083 -0.221  1.000 -0.637 -0.240
HR  0.391  0.528 -0.637  1.000  0.438
BB  0.252  0.301 -0.240  0.438  1.000


In [9]:
N_COMPONENTS = 3

pca = PCA(n_components=N_COMPONENTS, random_state=42)

# FIT + TRANSFORM on train
train_pca = pca.fit_transform(train[pca_features])

# TRANSFORM ONLY on test
test_pca = pca.transform(test[pca_features])

print(f'Explained variance ratio: {pca.explained_variance_ratio_.round(4)}')
print(f'Cumulative variance: {pca.explained_variance_ratio_.cumsum().round(4)}')

Explained variance ratio: [0.504  0.2539 0.1485]
Cumulative variance: [0.504  0.7579 0.9064]


In [10]:
# Add PCA components to DataFrames
pca_col_names = [f'hitting_pc{i+1}' for i in range(N_COMPONENTS)]

for i, col_name in enumerate(pca_col_names):
    train[col_name] = train_pca[:, i]
    test[col_name] = test_pca[:, i]

print(f'Added {len(pca_col_names)} PCA columns: {pca_col_names}')
print(f'\nTrain shape: {train.shape}')
print(f'Test shape:  {test.shape}')

# PCA loadings
loadings = pd.DataFrame(pca.components_.T, index=pca_features, columns=pca_col_names)
print(f'\nPCA Loadings:')
print(loadings.round(4))

Added 3 PCA columns: ['hitting_pc1', 'hitting_pc2', 'hitting_pc3']

Train shape: (1812, 82)
Test shape:  (453, 76)

PCA Loadings:
    hitting_pc1  hitting_pc2  hitting_pc3
H        0.4273       0.5872      -0.0914
2B       0.5150       0.3359      -0.2403
3B      -0.3382       0.6711       0.3071
HR       0.5404      -0.2770      -0.1456
BB       0.3818      -0.1238       0.9046


## Save Phase 2 Artifacts

In [11]:
# Save fitted unsupervised models (K-Means + PCA only — franchise agg is stateless)
joblib.dump(kmeans, os.path.join(ARTIFACT_DIR, 'kmeans.pkl'))
joblib.dump(pca, os.path.join(ARTIFACT_DIR, 'pca.pkl'))
print('Saved: artifacts/kmeans.pkl')
print('Saved: artifacts/pca.pkl')

# Update column info with new columns
col_info['cluster_ohe_cols'] = cluster_ohe_cols
col_info['pca_col_names'] = pca_col_names
col_info['cluster_features'] = cluster_features
col_info['pca_features'] = pca_features
col_info['n_clusters'] = N_CLUSTERS
col_info['n_pca_components'] = N_COMPONENTS
joblib.dump(col_info, os.path.join(ARTIFACT_DIR, 'column_info.pkl'))
print('Saved: artifacts/column_info.pkl (updated)')

# Save enriched DataFrames (Phase 2a: clusters + PCA, before franchise features)
train.to_csv(os.path.join(ARTIFACT_DIR, 'train_phase2.csv'), index=False)
test.to_csv(os.path.join(ARTIFACT_DIR, 'test_phase2.csv'), index=False)
print('Saved: artifacts/train_phase2.csv')
print('Saved: artifacts/test_phase2.csv')

# Summary before franchise section
all_feature_cols = feature_cols + cluster_ohe_cols + pca_col_names
print(f'\n=== Phase 2a Complete (Clusters + PCA) ===')
print(f'Total feature columns: {len(all_feature_cols)}')
print(f'  Base + Engineered: {len(feature_cols)}')
print(f'  K-Means OHE:       {len(cluster_ohe_cols)}')
print(f'  PCA Components:    {len(pca_col_names)}')
print(f'Train: {train.shape[0]} rows x {len(all_feature_cols)} features')
print(f'Test:  {test.shape[0]} rows x {len(all_feature_cols)} features')

Saved: artifacts/kmeans.pkl
Saved: artifacts/pca.pkl
Saved: artifacts/column_info.pkl (updated)
Saved: artifacts/train_phase2.csv
Saved: artifacts/test_phase2.csv

=== Phase 2a Complete (Clusters + PCA) ===
Total feature columns: 75
  Base + Engineered: 68
  K-Means OHE:       4
  PCA Components:    3
Train: 1812 rows x 75 features
Test:  453 rows x 75 features


## 4c. Franchise-Level Group-By Aggregation

Compute franchise historical statistics (mean, std, min, max) for key features.
These capture "team identity" — e.g., a franchise that historically scores lots of runs.

**Leakage check**: We aggregate only input features (not W), computed from the
combined train+predict raw data. This is safe because we're summarizing franchise
tendencies from observable stats, not from the target.

In [12]:
# Load raw data (pre-scaling) for franchise aggregation
raw_train = pd.read_csv('data/data_year_team_franchise.csv')
raw_predict = pd.read_csv('data/predict_year_team_franchise.csv')

# Combine for franchise-level stats (using only input features, NOT W)
raw_all = pd.concat([
    raw_train[['franchID', 'R', 'RA', 'HR', 'ERA', 'FP', 'BB', 'SO', 'E', 'ID']],
    raw_predict[['franchID', 'R', 'RA', 'HR', 'ERA', 'FP', 'BB', 'SO', 'E', 'ID']]
], ignore_index=True)

print(f"Combined raw data: {raw_all.shape}")
print(f"Unique franchises: {raw_all['franchID'].nunique()}")
print(f"\nRows per franchise:")
print(raw_all.groupby('franchID').size().describe())

Combined raw data: (2265, 10)
Unique franchises: 30

Rows per franchise:
count     30.000000
mean      75.500000
std       37.392374
min       19.000000
25%       45.000000
50%      106.000000
75%      108.750000
max      111.000000
dtype: float64


In [13]:
# Define aggregation targets — key performance indicators
agg_features = ['R', 'RA', 'HR', 'ERA', 'FP', 'E']
agg_funcs = ['mean', 'std']

# Compute franchise-level aggregates
franchise_agg = raw_all.groupby('franchID')[agg_features].agg(agg_funcs)
franchise_agg.columns = [f'franch_{feat}_{func}' for feat, func in franchise_agg.columns]
franchise_agg = franchise_agg.reset_index()

print(f"Franchise aggregation shape: {franchise_agg.shape}")
print(f"New columns ({len(franchise_agg.columns)-1}):")
for col in franchise_agg.columns[1:]:
    print(f"  {col}")
franchise_agg.head()

Franchise aggregation shape: (30, 13)
New columns (12):
  franch_R_mean
  franch_R_std
  franch_RA_mean
  franch_RA_std
  franch_HR_mean
  franch_HR_std
  franch_ERA_mean
  franch_ERA_std
  franch_FP_mean
  franch_FP_std
  franch_E_mean
  franch_E_std


,franchID,franch_R_mean,franch_R_std,franch_RA_mean,franch_RA_std,franch_HR_mean,franch_HR_std,franch_ERA_mean,franch_ERA_std,franch_FP_mean,franch_FP_std,franch_E_mean,franch_E_std
0,ANA,695.283019,102.818619,698.377358,80.799902,137.075472,36.945828,3.916792,0.499006,0.980208,0.003919,122.396226,25.471205
1,ARI,731.842105,70.085870,750.894737,76.687602,167.210526,24.663466,4.266842,0.430491,0.983421,0.002434,100.842105,14.641126
2,ATL,657.099099,99.510160,679.846847,95.701163,107.945946,62.557441,3.715946,0.580795,0.973360,0.009394,166.963964,61.433176
3,BAL,683.185185,102.628825,724.527778,134.961016,114.851852,60.985645,4.044444,0.885890,0.974917,0.010593,155.416667,69.951802
4,BOS,736.289720,110.234399,704.728972,100.683285,116.514019,63.819966,3.907664,0.704012,0.974570,0.008261,157.074766,53.735674


In [14]:
# Also compute "deviation from franchise mean" features
# These capture whether a team-season is above/below its franchise norm

# We need franchID on the phase2 train/test to merge
# Recover franchID from raw data via ID column
train_franchID = raw_train[['ID', 'franchID']].copy()
test_franchID = raw_predict[['ID', 'franchID']].copy()

# Merge franchID onto phase2 DataFrames
train_with_franch = train.merge(train_franchID, on='ID', how='left')
test_with_franch = test.merge(test_franchID, on='ID', how='left')

print(f"franchID coverage — train: {train_with_franch['franchID'].notna().sum()}/{len(train)}")
print(f"franchID coverage — test:  {test_with_franch['franchID'].notna().sum()}/{len(test)}")

franchID coverage — train: 1812/1812
franchID coverage — test:  453/453


In [15]:
# Merge franchise aggregates onto train and test
train_enriched = train_with_franch.merge(franchise_agg, on='franchID', how='left')
test_enriched = test_with_franch.merge(franchise_agg, on='franchID', how='left')

franch_agg_cols = [c for c in franchise_agg.columns if c != 'franchID']

print(f"Train enriched: {train_enriched.shape}")
print(f"Test enriched:  {test_enriched.shape}")
print(f"\nNew franchise columns: {franch_agg_cols}")

# Check for any NaNs introduced
print(f"\nNaN check (train): {train_enriched[franch_agg_cols].isnull().sum().sum()}")
print(f"NaN check (test):  {test_enriched[franch_agg_cols].isnull().sum().sum()}")

Train enriched: (1812, 95)
Test enriched:  (453, 89)

New franchise columns: ['franch_R_mean', 'franch_R_std', 'franch_RA_mean', 'franch_RA_std', 'franch_HR_mean', 'franch_HR_std', 'franch_ERA_mean', 'franch_ERA_std', 'franch_FP_mean', 'franch_FP_std', 'franch_E_mean', 'franch_E_std']

NaN check (train): 0
NaN check (test):  0


In [16]:
# Compute "deviation from franchise mean" for key raw stats
# Use raw values for R, RA, HR, ERA, E (from the raw CSVs, not scaled)
dev_features = ['R', 'RA', 'HR', 'ERA', 'E']

# Build deviation features using raw values
for feat in dev_features:
    mean_col = f'franch_{feat}_mean'
    dev_col = f'franch_{feat}_dev'
    
    # For train: raw value - franchise mean
    raw_vals_train = raw_train.set_index('ID')[feat]
    train_enriched[dev_col] = train_enriched['ID'].map(raw_vals_train) - train_enriched[mean_col]
    
    # For test: raw value - franchise mean  
    raw_vals_test = raw_predict.set_index('ID')[feat]
    test_enriched[dev_col] = test_enriched['ID'].map(raw_vals_test) - test_enriched[mean_col]

franch_dev_cols = [f'franch_{feat}_dev' for feat in dev_features]
print(f"Added deviation columns: {franch_dev_cols}")
print(f"\nTrain deviation stats:")
print(train_enriched[franch_dev_cols].describe().round(2))

Added deviation columns: ['franch_R_dev', 'franch_RA_dev', 'franch_HR_dev', 'franch_ERA_dev', 'franch_E_dev']

Train deviation stats:
       franch_R_dev  franch_RA_dev  franch_HR_dev  franch_ERA_dev  \
count       1812.00        1812.00        1812.00         1812.00   
mean           0.67           0.86           1.07            0.01   
std           98.66          98.35          53.76            0.65   
min         -326.41        -323.73        -127.51           -2.16   
25%          -61.36         -62.71         -36.97           -0.38   
50%            2.00          -1.36           5.33            0.02   
75%           62.27          65.18          39.10            0.44   
max          313.65         487.93         150.95            2.77   

       franch_E_dev  
count       1812.00  
mean          -0.40  
std           50.97  
min         -101.42  
25%          -34.29  
50%          -11.11  
75%           20.89  
max          260.36  


In [17]:
# Correlation of new franchise features with W (target)
all_franch_cols = franch_agg_cols + franch_dev_cols

corr_with_W = train_enriched[all_franch_cols + ['W']].corr()['W'].drop('W').sort_values(ascending=False)
print("Correlation of franchise features with W (wins):")
print("=" * 50)
for feat, corr in corr_with_W.items():
    marker = " ***" if abs(corr) > 0.1 else ""
    print(f"  {feat:25s} {corr:+.4f}{marker}")
print("\n*** = |corr| > 0.1 (potentially useful)")

Correlation of franchise features with W (wins):
  franch_R_dev              +0.5283 ***
  franch_HR_dev             +0.2947 ***
  franch_R_mean             +0.2246 ***
  franch_HR_mean            +0.1248 ***
  franch_R_std              +0.0738
  franch_FP_mean            +0.0697
  franch_HR_std             +0.0307
  franch_ERA_std            -0.0705
  franch_E_mean             -0.0714
  franch_E_std              -0.0764
  franch_FP_std             -0.0769
  franch_RA_std             -0.0906
  franch_ERA_mean           -0.1439 ***
  franch_RA_mean            -0.1728 ***
  franch_E_dev              -0.2959 ***
  franch_ERA_dev            -0.3908 ***
  franch_RA_dev             -0.4820 ***

*** = |corr| > 0.1 (potentially useful)


In [18]:
# Quick sanity check: do deviation features add info beyond the raw features?
# Compare R_dev correlation vs raw R correlation
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

target = train_enriched['W']

# Baseline: existing features only
baseline_feats = feature_cols + cluster_ohe_cols + pca_col_names
X_baseline = train_enriched[baseline_feats]

# With franchise features
X_with_franch = train_enriched[baseline_feats + all_franch_cols]

ridge = Ridge(alpha=1.0)
cv_baseline = cross_val_score(ridge, X_baseline, target, cv=5, scoring='neg_mean_absolute_error')
cv_franch = cross_val_score(ridge, X_with_franch, target, cv=5, scoring='neg_mean_absolute_error')

print(f"5-Fold MAE (Ridge, alpha=1.0):")
print(f"  Baseline (no franchise):  {-cv_baseline.mean():.4f} ± {cv_baseline.std():.4f}")
print(f"  + franchise features:     {-cv_franch.mean():.4f} ± {cv_franch.std():.4f}")
print(f"  Improvement:              {(-cv_baseline.mean()) - (-cv_franch.mean()):.4f}")

5-Fold MAE (Ridge, alpha=1.0):
  Baseline (no franchise):  2.7311 ± 0.0434
  + franchise features:     2.7351 ± 0.0445
  Improvement:              -0.0040


In [19]:
# Also test with ElasticNet (your current best model family)
from sklearn.linear_model import ElasticNet

en = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
cv_baseline_en = cross_val_score(en, X_baseline, target, cv=5, scoring='neg_mean_absolute_error')
cv_franch_en = cross_val_score(en, X_with_franch, target, cv=5, scoring='neg_mean_absolute_error')

print(f"5-Fold MAE (ElasticNet, alpha=0.1, l1=0.5):")
print(f"  Baseline (no franchise):  {-cv_baseline_en.mean():.4f} ± {cv_baseline_en.std():.4f}")
print(f"  + franchise features:     {-cv_franch_en.mean():.4f} ± {cv_franch_en.std():.4f}")
print(f"  Improvement:              {(-cv_baseline_en.mean()) - (-cv_franch_en.mean()):.4f}")

5-Fold MAE (ElasticNet, alpha=0.1, l1=0.5):
  Baseline (no franchise):  2.8440 ± 0.0702
  + franchise features:     2.9012 ± 0.0833
  Improvement:              -0.0572


## Save Phase 2b — With Franchise Features

Only save if the CV comparison above shows improvement. Otherwise, stick with Phase 2a outputs.

In [20]:
# Save franchise aggregation table + enriched DataFrames
franchise_agg.to_csv(os.path.join(ARTIFACT_DIR, 'franchise_agg.csv'), index=False)
print('Saved: artifacts/franchise_agg.csv')

# Drop franchID before saving (not a model feature)
train_final = train_enriched.drop(columns=['franchID'])
test_final = test_enriched.drop(columns=['franchID'])

train_final.to_csv(os.path.join(ARTIFACT_DIR, 'train_phase2b.csv'), index=False)
test_final.to_csv(os.path.join(ARTIFACT_DIR, 'test_phase2b.csv'), index=False)
print('Saved: artifacts/train_phase2b.csv')
print('Saved: artifacts/test_phase2b.csv')

# Update column info
col_info['franch_agg_cols'] = franch_agg_cols
col_info['franch_dev_cols'] = franch_dev_cols
joblib.dump(col_info, os.path.join(ARTIFACT_DIR, 'column_info.pkl'))

all_new_cols = all_franch_cols
print(f'\n=== Phase 2b Complete (+ Franchise Features) ===')
print(f'New franchise columns: {len(all_new_cols)}')
print(f'  Aggregates (mean/std): {len(franch_agg_cols)}')
print(f'  Deviations:            {len(franch_dev_cols)}')
print(f'Train: {train_final.shape}')
print(f'Test:  {test_final.shape}')

Saved: artifacts/franchise_agg.csv
Saved: artifacts/train_phase2b.csv
Saved: artifacts/test_phase2b.csv

=== Phase 2b Complete (+ Franchise Features) ===
New franchise columns: 17
  Aggregates (mean/std): 12
  Deviations:            5
Train: (1812, 99)
Test:  (453, 93)
